In [1]:
#| default_exp frida_old

In [2]:
#| hide
import nbdev; nbdev.nbdev_export()

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [4]:
#| export
from os import getenv
model_path = getenv("MODEL")

In [5]:
model_path = 'fred'

In [6]:
#| export
seq_length = 1024

full_path = f'./models/{model_path}'
import torch
from transformers import GPT2Tokenizer, T5ForConditionalGeneration 
tokenizer = GPT2Tokenizer.from_pretrained(full_path, eos_token='</s>')
model = T5ForConditionalGeneration.from_pretrained(full_path, torch_dtype=torch.bfloat16) 


In [7]:
model

T5ForConditionalGeneration(
  (shared): Embedding(50364, 1536)
  (encoder): T5Stack(
    (embed_tokens): Embedding(50364, 1536)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1536, out_features=1536, bias=False)
              (k): Linear(in_features=1536, out_features=1536, bias=False)
              (v): Linear(in_features=1536, out_features=1536, bias=False)
              (o): Linear(in_features=1536, out_features=1536, bias=False)
              (relative_attention_bias): Embedding(32, 24)
            )
            (layer_norm): FusedRMSNorm(torch.Size([1536]), eps=1e-06, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1536, out_features=4096, bias=False)
              (wi_1): Linear(i

In [8]:
#| export
model.to('cuda');
model.eval();

In [9]:
#| export
import deepspeed
model = deepspeed.init_inference(model, 
                                mp_size=1,
                                dtype=model.dtype,
                                replace_with_kernel_inject=True)


[2024-09-29 15:47:03,415] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2024-09-29 15:47:05,425] [INFO] [logging.py:96:log_dist] [Rank -1] DeepSpeed info: version=0.15.1, git-hash=unknown, git-branch=unknown
[2024-09-29 15:47:05,427] [WARNING] [config_utils.py:70:_process_deprecated_field] Config parameter mp_size is deprecated use tensor_parallel.tp_size instead
[2024-09-29 15:47:05,427] [INFO] [logging.py:96:log_dist] [Rank -1] quantize_bits = 8 mlp_extra_grouping = False, quantize_groups = 1


In [10]:
sum(p.numel() for p in model.parameters())

1740354048

In [11]:
#| export
from front.common import process_seq

def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool, temperature:float=0.5):
    lm_text = '<LM>' + prompt
    input_ids=torch.tensor([tokenizer.encode(lm_text)]).cuda()
    output_ids = model.generate(input_ids, do_sample=True, temperature=temperature, repetition_penalty=5.0, typical_p=0.9, top_k=10, top_p=0.95, #watermark=False,
                        max_new_tokens=length, 
                        num_return_sequences=num_samples,)

    result = [tokenizer.decode(o[1:]).replace('\n', ' ') for o in output_ids]
    result = process_seq(result)
    return result


In [13]:
%%time
get_sample('<LM>На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 1.33 s, sys: 192 ms, total: 1.52 s
Wall time: 1.52 s


[' – просто говно». –\xa0 И что же мне делать? ― спросил я. <...> Я ведь не Лев Толстой, а так себе… Просто говнюк какой-то! А ты говоришь «на словах»!',
 ' – просто говно. –\xa0 Это почему? ― спросил я, чувствуя себя немного неловко под взглядом двух пар глаз (одна из них была моей собственной).',
 ' – просто говно. –\xa0 А я, значит… Я тоже? Ну-ну! Что же ты молчишь тогда?.. Ты ведь знаешь ответ на этот вопрос... Знаешь?! Да что ж это такое!..',
 ' – просто дурак. –\xa0 А что, есть разница? ― спросил я с вызовом и даже немного обиделся на себя за это: неужто действительно так думаю про Толстого?..']

In [14]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 1.2 s, sys: 237 μs, total: 1.2 s
Wall time: 1.2 s


[' — просто мудак».  И я, конечно же не мог этого сказать.',
 ' — говно. —\xa0 А на деле ты дерьмо, потому что не умеешь читать и писать».',
 ' – просто говно. –\xa0 А ты, значит…? Ну-ну! И что же дальше будет с этим миром и людьми в нем?.. Я имею ввиду не только тебя лично... Но вообще всех людей на Земле?!',
 ' – говно. –\xa0 А на деле я просто Лев Толстой, и все! Я не могу быть говном… Ты что же думаешь?']

In [15]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 1.14 s, sys: 0 ns, total: 1.14 s
Wall time: 1.13 s


[' – говно. –\xa0 А на деле я, может быть… не знаю даже кто! Я вообще-то в Москве живу и работаю по специальности «менеджер».',
 ' – говно. –\xa0 А я и не Лев Толстой, а так… С другой стороны — кто же? Я просто человек со своей точкой зрения на жизнь».',
 ' – просто говно. –\xa0 Да, я знаю… Но мне нравится быть говном! Я люблю это ощущение свободы и легкости бытия!.. И еще меня привлекает в этом смысле слово «свобода».',
 ' — говно. —\xa0 Ну, не скажи… Я и сам иногда так думаю про себя».']